## Workbook containing the proposed figures for chapter 1 

Supplementary: report of NSE and KGE by model inputs

In [9]:
import numpy as np
import pandas as pd
from src.config import PROCESSED_DATA_DIR, OUTPUT_DATA_DIR, TEST_START_YEAR, TEST_END_YEAR

ENSEMBLE_MEMBERS = 10
model_list = ['area', 'baseline', 'topographic', 'phase-split']

# define the evaluation metrics (NSE, KGE, and PBIAS), robust to missing values (NaNs)
def nse(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    if not np.any(mask): return np.nan
    obs_masked = obs[mask]
    sim_masked = sim[mask]
    denom = np.sum((obs_masked - np.mean(obs_masked)) ** 2)
    if denom == 0: return np.nan
    return 1 - np.sum((obs_masked - sim_masked) ** 2) / denom

def kge(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    if not np.any(mask) or len(obs_masked := obs[mask]) < 2: return np.nan
    sim_masked = sim[mask]
    r = np.corrcoef(obs_masked, sim_masked)[0, 1]
    std_obs = np.std(obs_masked)
    mean_obs = np.mean(obs_masked)
    if std_obs == 0 or mean_obs == 0: return np.nan
    alpha = np.std(sim_masked) / std_obs
    beta = np.mean(sim_masked) / mean_obs
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

def pbias(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    if not np.any(mask): return np.nan
    obs_masked = obs[mask]
    denom = np.sum(obs_masked)
    if denom == 0: return np.nan
    return 100 * np.sum(sim[mask] - obs_masked) / denom

# load the ground truth data
static_atts = pd.read_csv(PROCESSED_DATA_DIR / "static_attributes.csv", index_col=0)
streamflow = pd.read_csv(PROCESSED_DATA_DIR / "combined_streamflow.csv", index_col=0)

# Convert the ground truth streamflow data to units of mm/day
streamflow.index = pd.to_datetime(streamflow.index)
test_years_mask = (streamflow.index.year >= TEST_START_YEAR) & (streamflow.index.year <= TEST_END_YEAR)
streamflow_test = streamflow.loc[test_years_mask]

# Align the static attributes order exactly with the streamflow columns
common_columns = streamflow_test.columns.intersection(static_atts.index)
streamflow_test = streamflow_test[common_columns]
areas = static_atts.loc[common_columns, 'basin_area_km2']

# Vectorized operation across all 290 columns at once
# (m3/s) / (km2 * 1e6) * 86400 s/day * 1000 mm/m -> mm/day
conversion_factor = 86400 * 1e3 / (areas * 1e6)
streamflow_mm = streamflow_test.multiply(conversion_factor, axis=1)

# Identify glaciated stations
glacier_col = [c for c in static_atts.columns if 'glacier_pct' in c.lower()][0]
glaciated_stations = static_atts[static_atts[glacier_col] > 0].index.intersection(streamflow_mm.columns)
all_stations = streamflow_mm.columns

# Container for final report records
summary_records = []

for model in model_list:
    # Storage arrays for metrics per member per station
    # Shape: (Ensemble_Members, Stations)
    member_kge_all = np.zeros((ENSEMBLE_MEMBERS, len(all_stations)))
    member_nse_all = np.zeros((ENSEMBLE_MEMBERS, len(all_stations)))
    member_pbias_all = np.zeros((ENSEMBLE_MEMBERS, len(all_stations)))
    
    # Track ensemble mean time-series prediction per station
    ens_mean_preds = pd.DataFrame(0.0, index=streamflow_mm.index, columns=all_stations)
    
    # 1. First Pass: Load all member predictions, calculate individual metrics, and build ensemble mean
    for i in range(ENSEMBLE_MEMBERS):
        file_fact = OUTPUT_DATA_DIR / model / f"{model}_preds_member_{i}.csv"
        # Assuming predictions columns match station IDs and rows match the date index
        preds = pd.read_csv(file_fact, index_col=0)
        preds.index = pd.to_datetime(preds.index)
        preds = preds.loc[streamflow_mm.index, all_stations] # Align perfectly
        
        ens_mean_preds += preds / ENSEMBLE_MEMBERS
        
        for s_idx, station in enumerate(all_stations):
            obs_series = streamflow_mm[station].values
            sim_series = preds[station].values
            member_kge_all[i, s_idx] = kge(obs_series, sim_series)
            member_nse_all[i, s_idx] = nse(obs_series, sim_series)
            member_pbias_all[i, s_idx] = pbias(obs_series, sim_series)

    # 2. Compute metrics for the actual Ensemble Mean time series
    ens_mean_kge_all = np.array([kge(streamflow_mm[st].values, ens_mean_preds[st].values) for st in all_stations])
    ens_mean_nse_all = np.array([nse(streamflow_mm[st].values, ens_mean_preds[st].values) for st in all_stations])
    ens_mean_pbias_all = np.array([pbias(streamflow_mm[st].values, ens_mean_preds[st].values) for st in all_stations])

    # 3. Aggregate metrics over spatial subsets (All vs. Glaciated)
    for subset_name, subset_stations in [("All Basins", all_stations), ("Glaciated Basins", glaciated_stations)]:
        # Get positional indices of subset stations relative to the 'all_stations' list
        sub_indices = [all_stations.get_loc(st) for st in subset_stations]
        
        # Individual member spatial aggregations
        sub_member_kge = np.nanmedian(member_kge_all[:, sub_indices], axis=1)
        sub_member_nse = np.nanmedian(member_nse_all[:, sub_indices], axis=1)
        sub_member_pbias = np.nanmedian(member_pbias_all[:, sub_indices], axis=1)
        
        sub_member_kge_gt07 = np.sum(member_kge_all[:, sub_indices] > 0.7, axis=1)
        sub_member_kge_lt0 = np.sum(member_kge_all[:, sub_indices] < 0.0, axis=1)
        
        # Ensemble mean spatial aggregations
        ens_mean_kge_med = np.nanmedian(ens_mean_kge_all[sub_indices])
        ens_mean_nse_med = np.nanmedian(ens_mean_nse_all[sub_indices])
        ens_mean_pbias_med = np.nanmedian(ens_mean_pbias_all[sub_indices])
        ens_mean_kge_gt07 = np.sum(ens_mean_kge_all[sub_indices] > 0.7)
        ens_mean_kge_lt0 = np.sum(ens_mean_kge_all[sub_indices] < 0.0)
        
        # Format strings as: Single Member Mean ± Std [Ensemble Mean Prediction value]
        def format_stat(member_arr, mean_val, is_int=False):
            fmt = "{:.0f}" if is_int else "{:.3f}"
            return f"{fmt.format(np.mean(member_arr))} ± {fmt.format(np.std(member_arr))} [{fmt.format(mean_val)}]"

        summary_records.append({
            "Model": model,
            "Subset": subset_name,
            "Median KGE": format_stat(sub_member_kge, ens_mean_kge_med),
            "Median NSE": format_stat(sub_member_nse, ens_mean_nse_med),
            "Median PBIAS (%)": format_stat(sub_member_pbias, ens_mean_pbias_med, is_int=False),
            "Stations KGE > 0.7": format_stat(sub_member_kge_gt07, ens_mean_kge_gt07, is_int=True),
            "Stations KGE < 0.0": format_stat(sub_member_kge_lt0, ens_mean_kge_lt0, is_int=True)
        })

# 4. Generate beautiful multi-index layout for clean terminal printing
df_res = pd.DataFrame(summary_records).set_index(["Subset", "Model"])

print("\n" + "="*95)
print("   STREAMFLOW MODEL BENCHMARK TEST SUMMARY")
print("   Format: Single Member Mean ± SD [Ensemble Mean Prediction Metric]")
print("="*95)
with pd.option_context('display.max_columns', None, 'display.width', 1000, 'display.colheader_justify', 'center'):
    print(df_res)
print("="*95 + "\n")


   STREAMFLOW MODEL BENCHMARK TEST SUMMARY
   Format: Single Member Mean ± SD [Ensemble Mean Prediction Metric]
                                   Median KGE             Median NSE             Median PBIAS (%)      Stations KGE > 0.7 Stations KGE < 0.0
Subset           Model                                                                                                                      
All Basins       area         0.491 ± 0.098 [0.526]  0.509 ± 0.075 [0.567]  -14.478 ± 10.332 [-12.535]     76 ± 16 [85]      30 ± 24 [17]   
Glaciated Basins area         0.767 ± 0.064 [0.794]  0.769 ± 0.064 [0.820]     -1.964 ± 5.470 [-1.232]     60 ± 11 [69]         0 ± 0 [0]   
All Basins       baseline     0.275 ± 0.302 [0.352]  0.349 ± 0.239 [0.453]  -31.906 ± 26.380 [-29.884]     49 ± 28 [44]      70 ± 64 [38]   
Glaciated Basins baseline     0.560 ± 0.290 [0.645]  0.570 ± 0.335 [0.735]  -18.232 ± 22.726 [-16.576]     40 ± 22 [37]        7 ± 16 [0]   
All Basins       topographic  0.688 ± 0.0

Sketch of counterfactual hypothesis

Table reporting of streamflow performance (NSE and KGE quartiles broken down by glaciation)

Streamflow model performance (NSE and KGE results)

SOM cluster patterns

Correlations and evaluation metrics

Supplementary: histogram of static attributes